## 0. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add langchain_community faiss-cpu langchain-ollama

### 2) OpenAI 인증키 설정
https://openai.com/

In [2]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:5])

gsk_D


In [ ]:
# ollama pull bge-m3

In [3]:
EMBEDDING_MODEL_NAME = "bge-m3:latest"
EMBEDDING_MODEL_NAME

'bge-m3:latest'

#### RAG 파이프 라인
* Load Data - Text Split - Indexing - Retrieval - Generation
* OllamaEmbeddings 사용

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_ollama import OllamaEmbeddings

from pprint import pprint

# 1. Load Data
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

print(type(documents), len(documents)) #[Document, Document]
print(type(documents[0]))
print(documents[0])

<class 'list'> 1
<class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.
    바. 대통령령으로 정하는 작물재배업에서 발생하는 소득
    사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발

In [5]:

# 2️. Text Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(documents)

print(len(split_docs), type(split_docs), type(split_docs[0]))
print(split_docs[0])
print('두번째 Document ====================')
print(split_docs[1])


12 <class 'list'> <class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득' metadata={'source': '../data/taxinfo.txt'}
두번째 Document ====================
page_content='가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서

In [6]:

# 3️. Indexing (벡터 저장)
embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)
# vector DB에 저장
vectorstore = FAISS.from_documents(
    documents=split_docs, 
    embedding=embeddings_model
)

# 로컬 파일로 저장
vectorstore.save_local("faiss_index_ollama")

In [7]:

# 4️. Retrieval (유사 문서 검색) k: 질의와 가장 유사한 문서(청크) 6개를 찾아 반환하기
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
# retriever변수는 VectoreStoreRetriever 객체이다.
print(type(retriever))
# **질문(쿼리)**에 대해 유사한 문서를 검색하는 역할
retrieved_docs = retriever.invoke("소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?")
print(type(retrieved_docs), len(retrieved_docs))
print(type(retrieved_docs[0]))
print(retrieved_docs[0])

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
<class 'list'> 6
<class 'langchain_core.documents.base.Document'>
page_content='제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득' metadata={'source': '../data/taxinfo.txt'}


In [8]:

# 5️. Generation (LLM 응답 생성)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    #model="meta-llama/llama-4-scout-17b-16e-instruct",
    model="moonshotai/kimi-k2-instruct-0905",
    temperature=0
)
# 유사도 검색의 결과를 str 문자열로 만들기
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
print(context)

제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31.>
1. 「공익신탁법」에 따른 공익신탁의 이익
2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득
    가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득

가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득
    나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.
    다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.
    바. 대통령령으로 정하는 작물재배업에서 발생하는 소득

바. 대통령령으로 정하는 작물재배업에서 발생하는 소득
    사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발생하는 소득
3. 근로소득과

In [9]:

response_context = llm.invoke(f"소득세법에서 비과세소득에 해당하는 소득은 무엇인가요? \
                              관련 정보: {context}")
print('context 적용한 결과')
pprint(response_context.content)

context 적용한 결과
('요약하자면, 「소득세법」 제12조(비과세소득)는  \n'
 '“소득이 있어도 국가가 소득세를 매기지 않는 소득”을 목록으로 정해 놓은 조문입니다. 2024년 1월 현재 조문‧시행령‧시행규칙을 모두 '
 '반영해 가장 자주 나오는 항목 중심으로 정리하면 다음과 같습니다. (※ 굵은 글씨는 실무·시험에서 출제 빈도가 높은 항목)\n'
 '\n'
 '1. 공익신탁 이익  \n'
 '   - 「공익신탁법」에 따라 공익사업용으로 신탁한 재산에서 나오는 소득\n'
 '\n'
 '2. 대표적인 사업소득 비과세 항목  \n'
 '   ① **논·밭을 작물 생산에 이용하게 함으로써 생기는 소득**  \n'
 '   ② 1세대 1주택 임대소득(기준시가 12억 이하 주택에 한정, 18년 이전 귀속분까지만 적용)  \n'
 '   ③ 농어가(농·어촌)부업소득 ※ 시행령이 정하는 일정 규모  \n'
 '   ④ **전통주 제조소득**  \n'
 '   ⑤ 5년 이상 조림 임지에서 벌채·양분해 생긴 소득(연 600만원 한도)  \n'
 '   ⑥ **기준 총수입 2천만원 이하 작물재배·어로·양식어업 소득**\n'
 '\n'
 '3. 대표적인 근로·퇴직소득 비과세 항목  \n'
 '   - **현역 병(兵) 급여**  \n'
 '   - 산재보험급여(요양·휴업·장해·간병·유족·장의 등 일체)  \n'
 '   - 「근로기준법」요양·휴업·장해·유족보상금  \n'
 '   - **종교인소득**  \n'
 '     - 교육자금, 식사(대), 실비변상비  \n'
 '     - 6세 이하 자녀 양육·출산비(월 20만원 한도)  \n'
 '     - 사택제공 이익  \n'
 '   - 산재연금, 국군포로연금 등\n'
 '\n'
 '4. 대표적인 기타소득 비과세 항목  \n'
 '   - 국가유공자 보훈급여·학습보조비, 북한이탈주민 정착금·보로금  \n'
 '   - **국가보안법 상금·보로금**  \n'
 '   - 훈장 부상(副賞) 및 대통령령이 정하는 상금 

In [10]:

response = llm.invoke(f"소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?")
print('context 적용하지 않은 결과')
pprint(response.content)


context 적용하지 않은 결과
('소득세법에서 **비과세소득**은 말 그대로 **소득세를 부과하지 않는 소득**을 말합니다. 즉, 이 소득은 세금 계산에서 제외됩니다.\n'
 '\n'
 '### ✅ 소득세법상 주요 비과세소득 예시\n'
 '다음은 소득세법 제4조 및 관련 법령에 따라 비과세되는 주요 소득들입니다:\n'
 '\n'
 '---\n'
 '\n'
 '### 1. **공동주택 임대소득** (소득세법 제6조의2)\n'
 '- **1세대 1주택**의 경우, **임대소득 합산액이 연 2천만원 이하**이면 비과세  \n'
 '- 단, 주택 수 및 거주 기간, 임대 기간 등 요건 충족 필요\n'
 '\n'
 '---\n'
 '\n'
 '### 2. **연금소득**\n'
 '- **국민연금, 공무원연금, 군인연금, 사학연금 등 법정연금**의 수령액  \n'
 '- 단, **일시금** 또는 **연금저축의 원리금**은 과세됨 (예: 연금저축은 비과세 아님)\n'
 '\n'
 '---\n'
 '\n'
 '### 3. **일정 한도 내 이자소득**\n'
 '- **주택청약종합저축 이자** (연 1천만원 한도)\n'
 '- **장기주택마련저축 이자** (연 3천만원 한도)\n'
 '- **청년우대형 청약통장 이자** (연 2천만원 한도)\n'
 '- **보증부사채 이자** 등\n'
 '\n'
 '---\n'
 '\n'
 '### 4. **정부 및 공공기관의 보조금, 장학금**\n'
 '- **국가, 지방자치단체, 공공기관**이 지급하는 **생활보조금, 장학금** 등  \n'
 '- 단, **기업 등 민간기관**이 지급하는 장학금은 과세됨\n'
 '\n'
 '---\n'
 '\n'
 '### 5. **상속·증여로 인한 소득**\n'
 '- **상속재산**, **증여재산**은 소득세법상 비과세  \n'
 '- (단, 증여세법 따로 적용 가능)\n'
 '\n'
 '---\n'
 '\n'
 '### 6. **명절선물품 및 복리후생비**\n'
 '- **근로복지기준법**에 따라

### 개선한 Source - version1
* Retriever 검색방법 개선
    * search_type="mmr",  # 최대 다양성 검색
    * search_kwargs={"k": 6, "fetch_k": 10}  

In [11]:

from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_ollama import OllamaEmbeddings

from pprint import pprint

# 1. 데이터 로드 (기존과 동일)
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

# 2. 텍스트 분할 개선
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 크기 증가
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],  # 자연스러운 분할을 위한 구분자
    length_function=len,
    is_separator_regex=False,
)
split_docs = splitter.split_documents(documents)

# 3. 인덱싱 (벡터 저장)
embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)

vectorstore = FAISS.from_documents(
    documents=split_docs, 
    embedding=embeddings_model
)
#vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())

vectorstore.save_local("faiss_index")

# 4. 검색 개선
"""
    최대 다양성 검색(Maximum Marginal Relevance, MMR)
    MMR은 유사도가 높은 문서를 찾는 것을 넘어, 유사도와 다양성이라는 두 가지 기준을 모두 고려함
    - search_type="mmr": 검색 방식을 MMR로 지정합니다.
    - fetch_k: 일차적으로 질의와 유사한 문서 10개를 벡터 저장소에서 가져옵니다.
    - k: fetch_k로 가져온 10개의 문서 중에서 최종적으로 6개를 선택합니다. 6개를 선택할 때, MMR 알고리즘은 다음 두 가지를 고려함
        : 질의와의 유사도가 높고, 이미 선택된 다른 문서들과의 유사도가 낮은 (즉, 내용이 다양한) 문서
    * MMR의 작동 원리:
    - 질의와 가장 유사한 fetch_k개(10개)의 문서를 예비 후보군으로 가져옵니다.
    - 이 후보군 중에서 질의와 가장 유사한 문서 하나를 첫 번째 결과로 선택합니다.
    - 남은 후보군 중에서 질의와의 유사도는 높으면서 (관련성), 
      이미 선택된 문서들과의 유사도는 낮은 (다양성) 문서를 찾아 다음 결과로 추가함
    - 이 과정을 k개(6개)의 문서가 모두 선택될 때까지 반복합니다.        
"""
retriever = vectorstore.as_retriever(
    search_type="mmr",  # 최대 다양성 검색
    search_kwargs={"k": 6, "fetch_k": 10}  # 더 많은 결과 검색
)

# 5. 프롬프트 엔지니어링
def generate_prompt(query, context):
    return f"""다음은 소득세법 비과세소득 관련 조항입니다. 문맥을 고려하여 질문에 답변하세요.

[관련 조항]
{context}

[질문]
{query}

[답변 요구사항]
- 비과세소득 유형을 계층적으로 구분하여 설명
- 각 항목별 구체적인 조건 명시
- 법조문의 항, 호, 목 번호를 포함
- 500자 이내로 간결하게 요약"""

# 검색 및 응답 생성
query = "소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?"
retrieved_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)  # 창의성 낮춤
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    #model="meta-llama/llama-4-scout-17b-16e-instruct",
    model="moonshotai/kimi-k2-instruct-0905",
    temperature=0
)
response = llm.invoke(generate_prompt(query, context))

print('개선된 결과:')
pprint(response.content)

개선된 결과:
('소득세법 제12조 비과세소득은 5 대 범주로 계층화됩니다.\n'
 '\n'
 '1. 공익신탁 이익(제1호)\n'
 '2. 사업소득 중  \n'
 ' －작물생산용 논·밭 임대(제2호 가목)  \n'
 ' －총수입 2천만원 이하 1주택 임대(2018년 이전)(나목)  \n'
 ' －5년 이상 조림지 600만원 이하 벌채·양도(마목)  \n'
 ' －대통령령이 정한 농어가부업, 전통주 제조, 작물재배, 어로·양식어업(다·라·바·사목)\n'
 '\n'
 '3. 근로·퇴직소득  \n'
 ' －병역급여, 국외근로금, 외국정부·국제기관 급여(차·파·하·거목)  \n'
 ' －산재·고용·공무원 등 법정 보험급여(다·라·마·바·사목)  \n'
 ' －월 20만원 한도 출산·보육금, 식사대(머목·러목)  \n'
 ' －직무발명보상금, 학자금(어·아목)\n'
 '\n'
 '4. 기타소득  \n'
 ' －보훈·탈북·국군포로 관련 급여(5호 가·나·마목)  \n'
 ' －국가지정문화재 서화·골동 양도익(바목)  \n'
 ' －종교관련종사자 학자금·식사·실비·보육(월 20만원)·사택(5호 아목)\n'
 '\n'
 '5. 기타  \n'
 ' －훈장 부상, 상훈금(5호 다목)  \n'
 ' －장학금 중 대학생 근로장학금(서목)')


### 개선한 Source - version2
* Prompt 개선

In [12]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader
from langchain_ollama import OllamaEmbeddings

from pprint import pprint

# 1. Load Data
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

print("=== 원본 문서 길이 ===")
print(f"전체 문서 길이: {len(documents[0].page_content)} 글자")

# 2. Text Split 개선
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]  # 법령 구조에 맞는 분리자
)
split_docs = splitter.split_documents(documents)

print(f"분할된 문서 수: {len(split_docs)}개")
print("=== 분할 예시 ===")
for i, doc in enumerate(split_docs[:3]):
    print(f"Chunk {i+1} ({len(doc.page_content)}글자): {doc.page_content[:100]}...")

# 3. Indexing
embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)

vectorstore = FAISS.from_documents(
    documents=split_docs, 
    embedding=embeddings_model
)

#vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())
vectorstore.save_local("faiss_index")

# 4. Retrieval 개선
retriever = vectorstore.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 6}  
)

query = "소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?"
retrieved_docs = retriever.invoke(query)

print(f"\n=== 검색된 문서 ({len(retrieved_docs)}개) ===")
for i, doc in enumerate(retrieved_docs):
    print(f"문서 {i+1}: {doc.page_content[:200]}...")
    print("---")

# 5. Generation - 개선된 프롬프트
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    #model="meta-llama/llama-4-scout-17b-16e-instruct",
    model="moonshotai/kimi-k2-instruct-0905",
    temperature=0
)
context = "\n\n".join([f"[문서 {i+1}]\n{doc.page_content}" for i, doc in enumerate(retrieved_docs)])

# 개선된 프롬프트 - 더 구체적인 지시사항
improved_prompt = f"""
당신은 세무 전문가입니다. 아래 소득세법 제12조 조항을 바탕으로 질문에 답변해주세요.

질문: {query}

법령 조항:
{context}

다음 형식으로 답변해주세요:
1. 비과세소득의 정의
2. 주요 비과세소득 항목들을 다음과 같이 분류:
   - 사업소득 관련
   - 근로소득/퇴직소득 관련  
   - 연금소득 관련
   - 기타소득 관련
3. 각 항목별 구체적인 조건이나 한도액 명시

답변은 법조문을 인용하면서 구체적으로 작성해주세요.
"""

# 비교용 - 기존 방식
simple_prompt = f"소득세법에서 비과세소득에 해당하는 소득은 무엇인가요? 관련 정보: {context}"

print("\n=== 개선된 프롬프트로 답변 ===")
response_improved = llm.invoke(improved_prompt)
pprint(response_improved.content)

print("\n" + "="*50)
print("=== 기존 프롬프트로 답변 ===")
response_simple = llm.invoke(simple_prompt)
pprint(response_simple.content)


=== 원본 문서 길이 ===
전체 문서 길이: 4971 글자
분할된 문서 수: 8개
=== 분할 예시 ===
Chunk 1 (738글자): 제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2....
Chunk 2 (636글자): 다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양...
Chunk 3 (792글자): 라. 「근로기준법」 또는 「선원법」에 따라 근로자ㆍ선원 및 그 유족이 받는 요양보상금, 휴업보상금, 상병보상금(傷病補償金), 일시보상금, 장해보상금, 유족보상금, 행방불명보상금, ...

=== 검색된 문서 (6개) ===
문서 1: 제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 2...
---
문서 2: 다. 대통령령으로 정하는 농어가부업소득
    라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득
    마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.
    바. 대통령령으로 정하는 작물재배업에서 발생하...
---
문서 3: 나. 「국가보안법」에 따라 받는 상금과 보로금
    다. 「상훈법」에 따른 훈장과 관련하여 받는 부상(副賞)이나 그 밖에 대통령령으로 정하는 상금과 부상
    라. 종업원등 또는 대학의 교직원이 퇴직한 후에 사용

In [ ]:

# 추가 개선: 다른 검색 방식 시도
print("\n" + "="*50)
print("=== 검색 방식 개선 테스트 ===")

# MMR(Maximum Marginal Relevance) 검색 - 다양성 확보
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20}
)
retrieved_docs_mmr = retriever_mmr.invoke(query)
context_mmr = "\n\n".join([f"[문서 {i+1}]\n{doc.page_content}" for i, doc in enumerate(retrieved_docs_mmr)])

response_mmr = llm.invoke(f"""
{query}

법령 조항 (MMR 검색):
{context_mmr}

위 법령을 바탕으로 비과세소득 항목들을 체계적으로 정리해주세요.
""")

print("=== MMR 검색 결과 ===")
pprint(response_mmr.content)